### Descarga de los datos

### By:
Bryan Escobar Restrepo

### Date:
2026-08-15

### Description:

Este cuaderno responde las siguientes preguntas que permiten entender el problema a solucionar en el trabajo propuesto:

- ¿Cual es el objetivo del problema?
- ¿Cómo se usará su solución?
- ¿Cuáles son las soluciones actuales (si las hay)?
- ¿Cómo se debe enmarcar este problema (supervisado / no supervisado, en línea / fuera de línea, etc.)
- ¿Cómo se debe medir el desempeño o el rendimiento de la solución, una primera intuicion?
- ¿La medida de desempeño está alineada con el objetivo del problema?
- ¿Cuál sería el desempeño o rendimiento mínimo necesario para alcanzar el objetivo del problema?
- ¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencias o herramientas ya creadas?
- ¿Hay experiencia del problema disponible?
(Importante) ¿Cómo se puede resolver el problema manualmente?
- Hacer un listado de los supuestos que hay hasta este momento.
- Cual es la fuente de los datos?
- Como se actualizan los datos?
- Cada cuanto tiempo se actualizan los datos


## 📚 Import  libraries

In [1]:
from pathlib import Path

import pandas as pd


## 💾 Load data

In [2]:
file_path = "/home/bryaner/Admisiones-project/data/01_raw/Admission_Predict.csv"
admisiones_df = pd.read_csv(file_path, low_memory=False)
admisiones_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GRE Score          612 non-null    float64
 1   TOEFL Score        604 non-null    float64
 2   University Rating  617 non-null    float64
 3   SOP                607 non-null    float64
 4   LOR                615 non-null    float64
 5   CGPA               621 non-null    float64
 6   Research           594 non-null    float64
 7   Chance of Admit    623 non-null    float64
dtypes: float64(8)
memory usage: 39.1 KB


In [3]:
admisiones_df.sample(10)

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
156,315.0,105.0,3.0,2.0,2.5,8.34,0.0,0.70
440,301.0,107.0,3.0,3.5,3.5,8.34,1.0,0.62
171,334.0,117.0,5.0,4.0,4.5,9.07,1.0,0.89
139,318.0,109.0,1.0,3.5,3.5,9.12,0.0,0.78
550,312.0,104.0,3.0,3.5,3.5,8.42,0.0,0.74
174,321.0,111.0,4.0,4.0,4.0,8.97,1.0,0.87
394,329.0,111.0,4.0,4.5,4.0,9.23,1.0,0.89
226,306.0,110.0,2.0,3.5,4.0,8.45,0.0,0.63
375,304.0,101.0,2.0,2.0,2.5,7.66,0.0,0.38
73,314.0,108.0,4.0,4.5,4.0,9.04,1.0,0.84


## 📊 Data info



In [4]:
admisiones_df.columns

Index(['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR ', 'CGPA',
       'Research', 'Chance of Admit '],
      dtype='str')

In [5]:
columns_to_use = ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR ', 'CGPA',
       'Research', 'Chance of Admit ']

## 💡 Preguntas y respuestas




# ¿Cual es el objetivo del problema?

Estimar, a partir del perfil académico de un aspirante, una puntuación continua en [0,1] que represente su probabilidad estimada de admisión a un programa de posgrado, para que el estudiante pueda priorizar y ordenar universidades. Ojo con el matiz: el objetivo de negocio real no es acertar el valor exacto, es ordenar correctamente las opciones y clasificarlas en "segura / probable / ambiciosa".

# ¿Cómo se usará la solución?

Como componente de scoring dentro de una herramienta de orientación: el estudiante introduce sus datos y recibe un score por universidad. Consumo bajo demanda (API REST) o por lotes. Es una salida orientativa, no decisoria; no hay automatización de una decisión con consecuencias irreversibles.

# ¿Cuáles son las soluciones actuales?

No hay sistema previo en este proyecto. Fuera de él: asesores académicos, estadísticas de admisión publicadas por las universidades, foros comunitarios (GradCafe, Yocket) y reglas heurísticas del tipo "GRE > 320 y GPA alto → apunta a top-10". Tratarlas como el status quo a superar.

# ¿Cómo se debe enmarcar este problema (supervisado / no supervisado, en línea / fuera de línea, etc.)?

Aprendizaje supervisado, regresión (target continuo y acotado), batch / offline.


# ¿Cómo se debe medir el desempeño o el rendimiento de la solución, una primera intuicion?

MAE como métrica principal: está en las mismas unidades del target, es directamente comunicable al usuario y es robusta.
RMSE como secundaria (penaliza errores grandes, relevantes si alguien descarta una universidad viable).
R² solo para comunicar.
Correlación de Spearman, porque el uso real es ordenar.

# ¿La medida de desempeño está alineada con el objetivo del problema?

Parcialmente. El MAE mide error de valor; el objetivo es ranking. Dos modelos con el mismo MAE pueden ordenar distinto. Por eso MAE + Spearman. Segundo desajuste: el target es una estimación subjetiva, no un resultado real de admisión, así que el error del modelo incluye el ruido de la propia etiqueta y no puede bajar indefinidamente.

# ¿Cuál sería el desempeño o rendimiento mínimo necesario para alcanzar el objetivo del problema?

El modelo debe superar la regla manual (0.0593, según la teoría).


# ¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencias o herramientas ya creadas?

credit scoring (probabilidad de impago), lead scoring, predicción de rendimiento académico. Reutilizable: sí se puede reutilizar.

# ¿Hay experiencia del problema disponible?

Sí, abundante. Este es un dataset público muy documentado con cientos de notebooks publicados.

# ¿Cómo se puede resolver el problema manualmente?

Se puede resolver con una suma ponderada de variables normalizadas

# ¿Cuál es la fuente de los datos?

corresponde al dataset público de admisiones de posgrado recopilado con perfiles de estudiantes  aspirando a másteres en EE. UU.

# ¿Cómo se actualizan los datos?

No existe mecanismo de actualización. Es un dataset estático.

# ¿Cada cuánto se actualizan?

No aplica. Ya que es un dataset estático.
